## EDA and cleaning data for silver layer

In [0]:
df = spark.sql("FROM supply_chain_live.bronze.raw_supply_chain")

df.display()

In [0]:
metadata = spark.sql("FROM supply_chain_live.bronze.metadata")

metadata.display()

In [0]:
print(df.columns)

In [0]:
df.select("Customer Email").limit(10).display()

In [0]:
df.select(
    "Customer Country",
    "Benefit per order",
    "Customer Email",
    "Shipping date (DateOrders)",
    "Order city"
).limit(10).display()

In [0]:
df.select("product Description", "Customer Email", "Order zipcode").distinct().display()

In [0]:
# Count the number off null values in each column

from pyspark.sql.functions import col, sum as spark_sum

# Convert the result to a dictionary so that we can loop through it
null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

# Only keep columns with null values
null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
import re

#     Customer Email    -> Customer_email
def to_snake_case(name):
    # [\s]+ = en eller flera mellanrum byts ut mot ett understreck
    return re.sub(r"[\s]+", "_", name.strip().casefold())

# All columns
def rename_column_to_snake_case(df):
    new_column = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_column)

to_snake_case("Customer     Email     AcCount")


In [0]:
df_clean_column = rename_column_to_snake_case(df)
df_clean_column.display()


In [0]:
df_clean_column.display()

In [0]:
# 6/19/2017 4:41

from pyspark.sql.functions import to_timestamp, col, coalesce, lit, when

df_clean = (
(
    df_clean_column.withColumn(
        "shipping_date", to_timestamp("shipping_date_(dateorders)", "M/d/yyyy H:m")
    )
    .withColumn(
        "order_zipcode", coalesce(col("order_zipcode").cast("string"), lit("unknown"))
    )
    .withColumn(
        "customer_zipcode",
        coalesce(col("customer_zipcode").cast("string"), lit("unknown")),
    )
    .withColumn(
        "customer_country",
        when(col("customer_country") == "EE. UU.", "United States").otherwise(
            col("customer_country"),
        ),
    ).withColumn(
        "order_date",
        to_timestamp("order_date_(dateorders)", "M/d/yyyy H:m"),
    )
).drop(
    "customer_email",
    "customer_password",
    "product_description"
)

)

df_clean.select("shipping_date", "order_date", "order_zipcode", "customer_zipcode").display()
